# Módulo 5 · Notebook 5 — Pandas DataFrame Agent (World Cup Matches)

En este notebook construimos un agente sobre un `DataFrame` de Pandas usando la integración de LangChain para tablas.

Dataset:
- `data/table/WorldCupMatches.csv`

Objetivo:
- hacer preguntas en lenguaje natural sobre los partidos de los Mundiales FIFA,
- y dejar que el agente ejecute operaciones de Pandas para responder.

## 1. Requisitos y advertencia de seguridad

Este agente usa internamente una tool de Python para ejecutar código generado por el LLM sobre el DataFrame.

> Usa este enfoque con datos confiables y en entornos controlados.

In [1]:
# Si hace falta, descomenta para instalar dependencias en el entorno del notebook.
# %pip install -U langchain langchain-openai langchain-experimental pandas python-dotenv

## 2. Imports y configuración

In [2]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
from langchain_openai import ChatOpenAI

BASE_DIR = Path("../../").resolve()
load_dotenv(BASE_DIR / ".env")

CSV_PATH = BASE_DIR / "data" / "table" / "WorldCupMatches.csv"
DEFAULT_GEMINI_MODEL = "gemini-2.0-flash"

/var/folders/t3/97hgmq6x6mg3dybs2fbsfcqr0000gn/T/ipykernel_13134/901056457.py:6: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
/Users/guane/Documentos/GlogalLogic/AI-course/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


## 3. Cargar tabla de partidos

In [3]:
df = pd.read_csv(CSV_PATH)

# Limpieza mínima útil para análisis
df["Attendance"] = pd.to_numeric(df["Attendance"], errors="coerce")
df["HomeTeamGoals"] = pd.to_numeric(df["HomeTeamGoals"], errors="coerce")
df["AwayTeamGoals"] = pd.to_numeric(df["AwayTeamGoals"], errors="coerce")
df["TotalGoals"] = df["HomeTeamGoals"].fillna(0) + df["AwayTeamGoals"].fillna(0)

print(f"Rows: {len(df)}")
print("Columnas:", list(df.columns))
display(df.head(3))

Rows: 836
Columnas: ['Year', 'Datetime', 'Stage', 'Stadium', 'City', 'HomeTeamName', 'HomeTeamGoals', 'AwayTeamGoals', 'AwayTeamName', 'WinConditions', 'Attendance', 'HalfTimeHomeGoals', 'HalfTimeAwayGoals', 'Referee', 'Assistant1', 'Assistant2', 'RoundID', 'MatchID', 'HomeTeamInitials', 'AwayTeamInitials', 'TotalGoals']


,Year,Datetime,Stage,Stadium,City,HomeTeamName,HomeTeamGoals,AwayTeamGoals,AwayTeamName,WinConditions,...,HalfTimeHomeGoals,HalfTimeAwayGoals,Referee,Assistant1,Assistant2,RoundID,MatchID,HomeTeamInitials,AwayTeamInitials,TotalGoals
0,1930,13 Jul 1930 - 15:00,Group 1,Pocitos,Montevideo,France,4,1,Mexico,,...,3,0,LOMBARDI Domingo (URU),CRISTOPHE Henry (BEL),REGO Gilberto (BRA),201,1096,FRA,MEX,5
1,1930,13 Jul 1930 - 15:00,Group 4,Parque Central,Montevideo,USA,3,0,Belgium,,...,2,0,MACIAS Jose (ARG),MATEUCCI Francisco (URU),WARNKEN Alberto (CHI),201,1090,USA,BEL,3
2,1930,14 Jul 1930 - 12:45,Group 2,Parque Central,Montevideo,Yugoslavia,2,1,Brazil,,...,2,0,TEJADA Anibal (URU),VALLARINO Ricardo (URU),BALWAY Thomas (FRA),201,1093,YUG,BRA,3


## 4. Configurar LLM (Gemini)

Usamos Gemini con endpoint OpenAI-compatible para mantener consistencia con tu proyecto.

In [5]:
gemini_api_openAI = os.getenv("OPENAI_API_KEY")
from langchain_ollama import OllamaEmbeddings, ChatOllama

llm = ChatOllama(model="llama3.2:1b", temperature=0)

print(f"LLM listo: {llm}")

LLM listo: output_version=None model='llama3.2:1b' temperature=0.0


## 5. Crear agente de DataFrame

Configuramos el agente en modo **tool-calling** para evitar errores de parseo tipo ReAct
(comunes con algunos modelos). Además activamos `handle_parsing_errors=True` para reintentos automáticos.

`allow_dangerous_code=True` habilita la ejecución de Python generado por el agente.
Es obligatorio en versiones recientes para dejar explícito el riesgo.

In [7]:
agent = create_pandas_dataframe_agent(
    llm,
    df,
    agent_type="tool-calling",
    verbose=True,
    allow_dangerous_code=True,
    agent_executor_kwargs={"handle_parsing_errors": True},
)

print("✅ Agente de Pandas creado (tool-calling)")

✅ Agente de Pandas creado


## 6. Consultas de ejemplo

In [8]:
queries = [
    "¿Cuántos partidos hay en la tabla?",
    "¿Cuál fue el partido con mayor asistencia y en qué año se jugó?",
    "Dame el top 5 de partidos con más goles totales (HomeTeamGoals + AwayTeamGoals).",
    "¿Cuál es el promedio de goles en partidos de Final?",
]

for q in queries:
    print("=" * 100)
    print("Pregunta:", q)
    out = agent.invoke({"input": q})
    # Compatibilidad con distintos formatos de salida
    answer = out.get("output", out) if isinstance(out, dict) else out
    print("Respuesta:", answer)


Pregunta: ¿Cuántos partidos hay en la tabla?


> Entering new AgentExecutor chain...


ValueError: An output parsing error occurred. In order to pass this error back to the agent and have it try again, pass `handle_parsing_errors=True` to the AgentExecutor. This is the error: Could not parse LLM output: `Thought: To find the number of rows in the dataframe, I should use the `len()` function.
Action: The input to the action is `df`.`
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 

## 7. Reto de clase (10-15 min)

Responde estas preguntas y valida con una celda de Pandas manual:

1. ¿Qué selección aparece más veces como local (`HomeTeamName`)?
2. ¿En qué ciudad hubo mayor asistencia acumulada?
3. ¿Cuál es el promedio de goles por década?

> Sugerencia: pide al agente que muestre la lógica de cálculo y luego verifica con Pandas.